# RAG Performance & Fairness Evaluation Toolkit (OpenVINO + LangChain)

This notebook demonstrates how to build and evaluate a Retrieval-Augmented Generation (RAG) pipeline using OpenVINO™ for accelerated performance on Intel hardware. We will use Hugging Face and LangChain libraries to construct the pipeline.

The process involves:
1.  **Environment Setup**: Installing necessary libraries.
2.  **LLM and Tokenizer Setup**: Loading a language model (Microsoft's Phi-3-mini) and its tokenizer, optimized with OpenVINO.
3.  **Embedding Model Setup**: Preparing an embedding model to convert text into vector representations.
4.  **Data Loading and Processing**: Fetching documents from a web source, splitting them into manageable chunks, and creating vector embeddings.
5.  **Vector Store and Retriever Setup**: Storing the embeddings in a ChromaDB vector store and setting up a retriever with reranking for improved accuracy.
6.  **Building the RAG Chain**: Creating a `RetrievalQA` chain that combines the retriever and the LLM.
7.  **Running the RAG Pipeline**: Asking a question to get a response from the RAG system.
8.  **Evaluation**: Using a comprehensive `OpenVINORAGEvaluator` to assess the quality of the generated response based on various metrics like BLEU, ROUGE, BERTScore, perplexity, and bias.

## 1. Environment Setup

First, let's ensure all the required Python packages are installed. The following commands handle the installation of essential libraries. These are typically only needed if you encounter version conflicts or issues with existing installations.

In [ ]:
# Execute only if needed (in case of any errors due to setuptools or whl installation)
!pip install setuptools==57.0.0 --force-reinstall
!pip install wheel==0.36.2 --force-reinstall
!pip uninstall comtypes -y
!pip install --no-cache-dir comtypes

## 2. LLM and Tokenizer Setup

Next, we load the Large Language Model (LLM) and its corresponding tokenizer. We use `optimum-intel` to convert and accelerate the model with OpenVINO. In this example, we use `microsoft/Phi-3-mini-4k-instruct`, but you can replace it with another compatible model.

- **`OVModelForCausalLM`**: Loads a causal language model and automatically converts it to the OpenVINO format (`export=True`).
- **`device="GPU"`**: Specifies that the model should run on the integrated GPU for acceleration. You can change this to `"CPU"`.

In [ ]:
from optimum.intel import OVModelForCausalLM
from transformers import AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

# Load model with OpenVINO backend
model = OVModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct", # You can plug in any other supported model
    export=True,  # Convert to OpenVINO format on the fly
    device="GPU"  # Specify GPU for inference, can also be "CPU"
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

### Create a LangChain-compatible LLM Pipeline

We now create a `text-generation` pipeline using the OpenVINO-optimized model and tokenizer. This pipeline is then wrapped in `HuggingFacePipeline` to make it compatible with the LangChain ecosystem. A quick test is run to confirm the pipeline is working correctly.

In [ ]:
# Create a text-generation pipeline with the OpenVINO model
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=model.device,
    max_new_tokens=100,
    top_k=50,
    temperature=0.1,
    do_sample=True
)

# Create a LangChain instance from the Hugging Face pipeline
llm = HuggingFacePipeline(pipeline=llm_pipeline)

# Test the pipeline with a sample query
response = llm.invoke("What is an ocean?")
print(response)

## 3. Embedding Model Setup

For the retrieval part of our RAG pipeline, we need an embedding model to convert text documents into numerical vectors. We use `OpenVINOBgeEmbeddings` from `langchain_community`, which provides OpenVINO-optimized embeddings for efficient performance. Here, we use the `bge-small-en-v1.5` model.

In [ ]:
from langchain_community.embeddings import OpenVINOBgeEmbeddings

embedding_model_name = "bge-small-en-v1.5" # You can plug in any other embedding model
embedding_model_kwargs = {"device": "CPU"} # Embeddings often run well on CPU
encode_kwargs = {
    "normalize_embeddings": True,
}

# Initialize OpenVINO-optimized embeddings
embedding = OpenVINOBgeEmbeddings(
    model_name=embedding_model_name,
    model_kwargs=embedding_model_kwargs,
    encode_kwargs=encode_kwargs,
)

# Sample text to verify embedding functionality
text = "This is a test document."
embedding_result = embedding.embed_query(text)
print("Sample embedding (first 3 dimensions):", embedding_result[:3])

## 4. Data Loading and Processing

Now we'll load the documents that will form the knowledge base for our RAG pipeline. This notebook includes two methods for loading documents:

1.  **Web Crawling (Enabled by default)**: Fetches content from a website's sitemap. We use `WebBaseLoader` to load content from URLs found in the sitemap of Zerodha Varsity.
2.  **Local File Loading (Commented out)**: A robust `LangChainDocumentLoader` class is provided to load various file types (`.txt`, `.pdf`, `.docx`, etc.) from a local directory. You can uncomment and adapt this section if you want to use your own local files.

In [ ]:
import bs4
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup
import ssl
from langchain_community.document_loaders import WebBaseLoader

# --- Method 1: Load documents by crawling a web page (default) ---
def get_sitemap(url):
    """Fetches and parses an XML sitemap from a URL."""
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    response = urlopen(req)
    xml = BeautifulSoup(response, "lxml-xml", from_encoding=response.info().get_param("charset"))
    return xml

def get_urls_from_sitemap(xml):
    """Extracts all URLs from a parsed sitemap XML."""
    urls = [loc.text for loc in xml.find_all("loc")]
    return urls

# Bypass SSL verification issues if they arise
ssl._create_default_https_context = ssl._create_stdlib_context

sitemap_url = "https://zerodha.com/varsity/chapter-sitemap2.xml"
sitemap_xml = get_sitemap(sitemap_url)
urls = get_urls_from_sitemap(sitemap_xml)

# Load documents from the collected URLs
docs = []
for i, url in enumerate(urls):
    try:
        loader = WebBaseLoader(url)
        docs.extend(loader.load())
        if (i + 1) % 10 == 0:
            print(f"Loaded {i + 1}/{len(urls)} URLs")
    except Exception as e:
        print(f"Failed to load {url}: {e}")

print(f"\nTotal documents loaded: {len(docs)}")

# --- Method 2: Load documents locally from the system (commented out) ---
'''
import os
from langchain.document_loaders import (
    TextLoader,
    PyPDFLoader,
    DirectoryLoader,
)
from langchain.schema import Document as LCDocument
from typing import List

class LocalDocumentLoader:
    """Load documents from a local directory using LangChain loaders."""
    def __init__(self, directory_path: str):
        self.directory_path = directory_path

    def load(self) -> List[LCDocument]:
        """Loads all supported documents from the directory."""
        if not self.directory_path:
            raise ValueError("Directory path not set.")

        # Define loaders for different file types
        txt_loader = DirectoryLoader(
            self.directory_path, glob="**/*.txt", loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"}, show_progress=True
        )
        pdf_loader = DirectoryLoader(
            self.directory_path, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True
        )

        documents = []
        documents.extend(txt_loader.load())
        documents.extend(pdf_loader.load())
        
        return documents

# Usage Example:
# loader = LocalDocumentLoader(directory_path="/path/to/your/documents")
# docs = loader.load()
# print(f"Loaded {len(docs)} local documents.")
'''

### Split Documents into Chunks

LLMs have a limited context window, so we need to split large documents into smaller chunks. This ensures that the model can process the retrieved information effectively. We use `RecursiveCharacterTextSplitter` which is a smart way to split text while trying to keep related content together.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Split the documents into smaller chunks with a specified size and overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1250,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False
)

split_docs = text_splitter.split_documents(docs)
print(f"Documents split into {len(split_docs)} chunks.")

## 5. Vector Store and Retriever Setup

Now we'll create a vector store to house the document embeddings and enable efficient similarity searches.

- **`Chroma`**: We use ChromaDB as our vector store. It's a lightweight and easy-to-use vector database.
- **`persist_directory`**: This saves the created database to disk, allowing us to reuse it later without re-processing the documents.

In [ ]:
# Create a ChromaDB instance to store the document embeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    embedding_function=embedding,
    persist_directory="./chromadb_varsity",
    collection_name="zerodha_varsity_docs"
)

### Add Documents to the Vector Store

We add the processed document chunks to the vector store. To handle a large number of documents efficiently, we add them in batches. The metadata is also filtered to ensure compatibility with the vector store.

In [ ]:
from langchain_community.vectorstores.utils import filter_complex_metadata

# Function to insert embeddings in batches for a lengthy document set
def add_documents_in_batches(vectorstore, docs, batch_size=100):
    """Adds documents to the vectorstore in batches."""
    for i in range(0, len(docs), batch_size):
        chunk = docs[i : i + batch_size]
        vectorstore.add_documents(chunk)
        print(f"Added batch {i//batch_size + 1}/{(len(docs)-1)//batch_size + 1}")
    # Persist the database to disk if the method is available
    if hasattr(vectorstore, "persist"):
        vectorstore.persist()

# Filter out complex metadata that might cause issues
filtered_docs = filter_complex_metadata(split_docs)

# Add the documents to the vector store in batches
add_documents_in_batches(vectorstore, filtered_docs)

### Set up a Reranking Retriever

To improve the quality of retrieved documents, we use a reranker. The initial retriever fetches a set of documents (e.g., k=5), and the reranker (`FlashrankRerank`) re-orders them based on their relevance to the query. This ensures that the most relevant context is passed to the LLM.

- **`ContextualCompressionRetriever`**: Wraps a base retriever and a document compressor (the reranker) to create this two-stage retrieval process.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank

# Set up the base retriever to fetch the top 5 documents
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Initialize the reranker
compressor = FlashrankRerank()

# Create the compression retriever, which combines retrieval and reranking
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

## 6. Building the RAG Chain

With all the components ready, we now assemble the final RAG pipeline using LangChain's `RetrievalQA` chain. This chain connects the LLM with the retriever.

- **`chain_type="stuff"`**: This means all retrieved documents will be "stuffed" into the prompt sent to the LLM.
- **`return_source_documents=True`**: This is important for evaluation, as it allows us to see which documents were used to generate the answer.

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True
)

## 7. Running the RAG Pipeline

It's time to ask a question! The `qa_chain.invoke` method will execute the full RAG process: retrieve relevant documents, pass them to the LLM along with the question, and return the final answer.

In [ ]:
question = "What is a mutual fund?"
result = qa_chain.invoke({"query": question})
print("--- Question ---")
print(question)
print("\n--- Answer ---")
print(result["result"])

### Extract Answer and Context for Evaluation

For the evaluation step, we need to isolate the generated answer and the source documents (the context or "reference").

In [ ]:
answer = result['result']
context = " ".join([d.page_content for d in result['source_documents']])

## 8. Evaluation

To assess the quality of our RAG pipeline, we use a custom `OpenVINORAGEvaluator` class. This class uses OpenVINO-optimized models to calculate several key metrics:

- **BLEU & ROUGE**: Measure the overlap between the generated answer and the reference context.
- **BERTScore**: Computes semantic similarity, which is more advanced than simple overlap.
- **Perplexity**: Measures how well a language model (here, Llama-2-7B) predicts the generated text. Lower is better.
- **Diversity**: Calculates the variety of tokens in the response.
- **Racial Bias**: Uses a hate speech detection model to check for biased content.

**Note**: The first time you run this, it will download and convert the necessary evaluation models (Llama-2-7B and a hate speech model) to the OpenVINO format. This is a one-time setup.

In [ ]:
import openvino as ov
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from optimum.intel import OVModelForCausalLM, OVModelForSequenceClassification
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score
from nltk.util import ngrams
from typing import List
import os

class OpenVINORAGEvaluator:
    """An evaluator for RAG pipelines using OpenVINO-optimized models."""
    
    def __init__(self, device="GPU", models_dir="./openvino_models"):
        self.device = device
        self.models_dir = models_dir
        os.makedirs(self.models_dir, exist_ok=True)
        
        # Initialize models and tokenizers for evaluation
        self.llama2_model, self.llama2_tokenizer = self._load_model(
            model_id="meta-llama/Llama-2-7b-hf",
            ov_model_class=OVModelForCausalLM,
            subfolder="llama2-7b-openvino"
        )
        self.bias_model, self.bias_tokenizer = self._load_model(
            model_id="Hate-speech-CNERG/dehatebert-mono-english",
            ov_model_class=OVModelForSequenceClassification,
            subfolder="hate-speech-openvino"
        )
        print(f"OpenVINO RAG Evaluator initialized on {device}")

    def _load_model(self, model_id, ov_model_class, subfolder):
        """Generic function to load or convert a model to OpenVINO format."""
        model_path = os.path.join(self.models_dir, subfolder)
        
        if not os.path.exists(os.path.join(model_path, "openvino_model.xml")):
            print(f"Converting {model_id} to OpenVINO format...")
            ov_model = ov_model_class.from_pretrained(model_id, export=True, compile=False)
            ov_model.save_pretrained(model_path)
            print(f"Model saved to {model_path}")
        
        try:
            print(f"Loading {model_id} from {model_path}...")
            model = ov_model_class.from_pretrained(model_path, device=self.device)
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            print(f"{model_id} loaded successfully.")
            return model, tokenizer
        except Exception as e:
            print(f"Error loading {model_id}: {e}")
            return None, None

    def evaluate_bleu_rouge(self, candidates: List[str], references: List[str]):
        """Calculates BLEU and ROUGE scores."""
        candidate_tokens = [c.split() for c in candidates]
        reference_tokens = [[r.split()] for r in references]
        
        # BLEU with smoothing
        smoothing = SmoothingFunction().method1
        bleu_score = corpus_bleu(reference_tokens, candidate_tokens, smoothing_function=smoothing)
        
        # ROUGE
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
        rouge1_f1 = sum(scorer.score(ref, cand)['rouge1'].fmeasure for ref, cand in zip(references, candidates)) / len(candidates)
        return bleu_score, rouge1_f1

    def evaluate_bert_score(self, candidates: List[str], references: List[str]):
        """Calculates BERTScore."""
        _, _, f1 = score(candidates, references, lang="en", model_type='bert-base-multilingual-cased')
        return f1.mean().item()

    def evaluate_perplexity(self, text: str):
        """Calculates perplexity using the loaded Llama-2 model."""
        if not self.llama2_model:
            return float('inf')
        
        encodings = self.llama2_tokenizer(text, return_tensors='pt', max_length=1024, truncation=True)
        input_ids = encodings.input_ids.to(self.llama2_model.device)
        
        with torch.no_grad():
            outputs = self.llama2_model(input_ids, labels=input_ids)
            loss = outputs.loss
            perplexity = torch.exp(loss)
        
        return perplexity.item()

    def evaluate_racial_bias(self, text: str):
        """Evaluates racial bias using a hate speech detection model."""
        if not self.bias_model:
            return 0.0

        inputs = self.bias_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits = self.bias_model(**inputs).logits
            probabilities = torch.nn.functional.softmax(logits, dim=-1)
            # Return the probability of the 'hate speech' class (index 1)
            bias_score = probabilities[0][1].item()
        return bias_score
    
    def evaluate_all(self, response: str, reference: str):
        """Runs a comprehensive evaluation and returns all metrics."""
        candidates = [response]
        references = [reference]
        
        try:
            bleu, rouge1 = self.evaluate_bleu_rouge(candidates, references)
            bert_f1 = self.evaluate_bert_score(candidates, references)
            perplexity = self.evaluate_perplexity(response)
            racial_bias = self.evaluate_racial_bias(response)
            
            return {
                "BLEU": bleu,
                "ROUGE-1": rouge1,
                "BERT F1": bert_f1,
                "Perplexity": perplexity,
                "Racial Bias": racial_bias
            }
        except Exception as e:
            print(f"An error occurred during evaluation: {e}")
            return {k: 0.0 for k in ["BLEU", "ROUGE-1", "BERT F1", "Perplexity", "Racial Bias"]}

### Run the Evaluation

Finally, we initialize the `OpenVINORAGEvaluator` and call `evaluate_all` to get a dictionary of scores. This provides a quantitative look at the performance of our RAG pipeline for the given query.

In [ ]:
# Initialize the evaluator (this might take a moment on the first run)
evaluator = OpenVINORAGEvaluator(device="GPU")

# Prepare the data for evaluation
response_text = answer
reference_text = context.strip()

# Get all evaluation metrics
metrics = evaluator.evaluate_all(response_text, reference_text)

In [ ]:
print("--- Evaluation Metrics ---")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")